<a href="https://colab.research.google.com/github/tiagomelo33/deeplearningandmachine/blob/main/CINZAS_TOTAIS_PREDI%C3%87%C3%83O.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================
# Algoritmo para prever Cinzas Totais (t)
# Fórmula invertida + Rede Neural com aprendizado contínuo
# ============================================

import os
import pickle
import numpy as np
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler

# Caminhos para salvar modelo e scaler
MODEL_PATH = 'modelo_cinzas.keras'
SCALER_PATH = 'scaler.pkl'
DATA_PATH = 'dados_cinzas.npy'

# Fórmula invertida para Cinzas totais
def calcular_cinzas(vida_util, cinzas_th, torque):
    A = 2091.7335
    b = -0.3945
    c = -0.2226
    d = -0.1726
    return ((vida_util)/(A*(cinzas_th**c)*(torque**d)))**(1/b)

# Inicializar ou carregar modelo e scaler
def inicializar_modelo():
    if os.path.exists(MODEL_PATH) and os.path.exists(SCALER_PATH):
        model = load_model(MODEL_PATH)
        with open(SCALER_PATH,'rb') as f:
            scaler = pickle.load(f)
        return model, scaler
    # Se não existir, cria modelo com dados sintéticos
    rng = np.random.default_rng(0)
    dados = np.column_stack([
        rng.uniform(40, 120, 200),  # vida útil
        rng.uniform(0.3, 1.5, 200), # cinzas t/h
        rng.uniform(30, 60, 200)    # torque
    ])
    cinzas_real = rng.uniform(500, 2000, 200)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(dados)
    model = Sequential([
        Dense(32, activation='relu', input_shape=(3,)),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(Xs, cinzas_real, epochs=100, batch_size=16, verbose=0)
    model.save(MODEL_PATH)
    with open(SCALER_PATH,'wb') as f:
        pickle.dump(scaler,f)
    return model, scaler

# Salvar dados para treino incremental
def salvar_dados(novo_dado):
    if os.path.exists(DATA_PATH):
        dados_existentes = np.load(DATA_PATH)
        dados_novos = np.vstack([dados_existentes, novo_dado])
    else:
        dados_novos = novo_dado
    np.save(DATA_PATH, dados_novos)

# Treinar incrementalmente
def treinar_incremental(model, scaler, epochs=10):
    dados = np.load(DATA_PATH)
    X = dados[:, :3]
    y = dados[:, 3]
    Xs = scaler.transform(X)
    model.fit(Xs, y, epochs=epochs, verbose=0)
    model.save(MODEL_PATH)
    return model

# Prever Cinzas totais
def prever_cinzas(vida_util, cinzas_th, torque, model, scaler):
    formula_res = calcular_cinzas(vida_util, cinzas_th, torque)
    x = np.array([[vida_util, cinzas_th, torque]])
    xs = scaler.transform(x)
    nn_res = float(model.predict(xs, verbose=0)[0][0])
    return formula_res, nn_res

# ============================================
# Loop interativo no Colab
# ============================================

model, scaler = inicializar_modelo()

while True:
    try:
        vida_util = float(input("\nInforme VIDA ÚTIL (dias): "))
        cinzas_th = float(input("Informe CINZAS t/h (t/h): "))
        torque = float(input("Informe TORQUE (%): "))
    except ValueError:
        print("Valores inválidos. Tente novamente.")
        continue

    # Previsão
    formula_res, nn_res = prever_cinzas(vida_util, cinzas_th, torque, model, scaler)
    print(f"\n→ Cinzas totais pela fórmula: {formula_res:.2f} t")
    print(f"→ Cinzas totais pela rede neural: {nn_res:.2f} t")

    # Registrar dado real
    resp = input("\nRegistrar CINZAS REAIS para aprendizado? (s/n): ").strip().lower()
    if resp == 's':
        try:
            cinzas_real = float(input("Informe Cinzas reais (t): "))
            novo_dado = np.array([[vida_util, cinzas_th, torque, cinzas_real]])
            salvar_dados(novo_dado)
            model = treinar_incremental(model, scaler)
            print("Modelo atualizado com o novo exemplo.")
        except ValueError:
            print("Valor inválido.")

    if input("\nContinuar? (s/n): ").strip().lower() != 's':
        print("Saindo. Modelo salvo.")
        break



Informe VIDA ÚTIL (dias): 90
Informe CINZAS t/h (t/h): 1.1
Informe TORQUE (%): 55

→ Cinzas totais pela fórmula: 476.97 t
→ Cinzas totais pela rede neural: 1144.10 t

Registrar CINZAS REAIS para aprendizado? (s/n): 478

Continuar? (s/n): N
Saindo. Modelo salvo.
